In [ ]:
import sys
from pathlib import Path

import os
import sys
from pathlib import Path
import json
import warnings
import shutil
import random
import string

import torch
from torch.nn import functional as F
import nibabel as nib
import pandas as pd
import numpy as np
import pytorch_lightning as pl
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, r2_score
from scipy.stats import pearsonr, spearmanr

from lit_classifier import LitClassifier
from datasets import NiftiDataset, nifti_loader
from utils import confusion_matrix_to_figure
from transformations import get_transforms
from utils_hparams import args_short_to_long

warnings.filterwarnings("ignore")

seed = 1234
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True  # CuDNN deterministic
torch.backends.cudnn.benchmark = False  # CuDNN benchmark off
torch.random.manual_seed(seed)
pl.seed_everything(seed)

In [ ]:
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


best_weights_path = '$PATH/RI_anatomical_prior/log/basic_0619/t1/version_3/checkpoints/epoch=023.ckpt'

def _add_missing_hparams(model):
    hparams_all = {v[0]: v[0] for k, v in args_short_to_long.items()}  # transform to dict param_long_name:default_value
    for param, default_value in hparams_all.items():
        if param not in model.hparams:
            if "lightning_quiet" not in os.environ:
                print(f"Missing hparam '{param}' in stored model. Adding with value {default_value}.")
            model.hparams[param] = default_value
    return model


batch_size = 70
device = torch.device('cuda:0')

models = []
model = LitClassifier.load_from_checkpoint(best_weights_path)
model = _add_missing_hparams(model)
model.eval().to(device)
models.append(model)
model = models[0] 

subjects_test = model.hparams.subjects_test
labels_test = model.hparams.labels_test
wm_test = model.hparams.wm_test
icv_test = model.hparams.icv_test
ratio_test = model.hparams.ratio_test

data_path = Path('$PATH/datasets/infancy_brain_MRI')

if hasattr(model.hparams, "img_files"):
    img_files = model.hparams.img_files  
else:
    config = json.load(open(data_path / "dataset.json"))
    config["img_files"]
img_files = [f.replace(".npy", ".nii.gz") for f in img_files]

tfs_train, tfs_val = get_transforms(model.hparams)
tfs_used = tfs_val
tta_nr = 1

dataset_val = NiftiDataset(data_path, subjects_test, labels_test, wm_test, icv_test, ratio_test,
                            transform=tfs_used,
                            dim=model.hparams.dim,
                            nr_slices=model.hparams.nr_slices,
                            multi_orientation=model.hparams.multi_orientation,
                            crop_size=model.hparams.crop_size,
                            slice_subset=model.hparams.slice_subset,
                            img_files=img_files,
                            loss=model.hparams.loss,
                            nr_classes=model.hparams.nr_classes,
                            deterministic=True,
                            tiles=model.hparams.tiles,
                            file_loader=nifti_loader,
                            tta_nr=tta_nr,
                            tiles_subsample=model.hparams.tiles_subsample,
                            tiles_start=model.hparams.tiles_start,
                            zoom=model.hparams.zoom,
                            slice_orientation=model.hparams.slice_orientation)

# Per default DataLoader not doing any shuffling even with num_workers > 1
loader_val = DataLoader(dataset_val, batch_size=batch_size, num_workers=10, pin_memory=True)

data = next(iter(loader_val))
samples, y, a, b, c = data

encoder = model.backbone
layer_names = ['Layer1', 'Layer2', 'Layer3']
layers = [encoder.Conv_1, encoder.Conv_2, encoder.Conv_3]

i = 69
sample = samples[i]
wm = torch.tensor(a[i].reshape(-1, 1).float()).to(device)
icv = torch.tensor(c[i].reshape(-1, 1).float()).to(device)
info = torch.concat((wm, icv), dim=1)

slice_idx = 60
input_tensor = torch.tensor(sample).unsqueeze(0).to(device) 
input_for_viz = input_tensor.squeeze(0).permute(1, 2, 3, 0).cpu().numpy() 

img_slice1 = input_for_viz[:, :, slice_idx, 0:1]
img_slice = img_slice1
img_norm = (img_slice - img_slice.min()) / (img_slice.max() - img_slice.min() + 1e-6)

# figure setup
fig, axes = plt.subplots(1, len(layers) + 1, figsize=(3 * (len(layers) + 1), 4))

fig.text(0.02, 0.5, 'T1w', va='center', rotation='vertical', fontsize=25)

axes[0].imshow(np.rot90(img_norm), cmap='gray')
axes[0].set_title("Original", fontsize=25)
axes[0].axis("off")

for i, layer in enumerate(layers):
    cam = GradCAM(model=model, target_layers=[layer])
    grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0]  # [D, H, W]
    cam_slice = grayscale_cam[:, :, slice_idx]
    overlay = show_cam_on_image(img_norm, cam_slice, use_rgb=True, image_weight=0.05)

    axes[i + 1].imshow(np.rot90(overlay), cmap='jet')
    axes[i + 1].set_title(layer_names[i], fontsize=25)
    axes[i + 1].axis("off")

plt.tight_layout(rect=[0.04, 0, 1, 1])
plt.savefig(f"T1w.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close() 

In [ ]:
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


best_weights_path = '$PATH/RI_anatomical_prior/log/basic_0619/t2/version_3/checkpoints/epoch=020.ckpt'

def _add_missing_hparams(model):
    hparams_all = {v[0]: v[0] for k, v in args_short_to_long.items()}  # transform to dict param_long_name:default_value
    for param, default_value in hparams_all.items():
        if param not in model.hparams:
            if "lightning_quiet" not in os.environ:
                print(f"Missing hparam '{param}' in stored model. Adding with value {default_value}.")
            model.hparams[param] = default_value
    return model


batch_size = 70
device = torch.device('cuda:0')

models = []
model = LitClassifier.load_from_checkpoint(best_weights_path)
model = _add_missing_hparams(model)
model.eval().to(device)
models.append(model)
model = models[0] 

subjects_test = model.hparams.subjects_test
labels_test = model.hparams.labels_test
wm_test = model.hparams.wm_test
icv_test = model.hparams.icv_test
ratio_test = model.hparams.ratio_test

data_path = Path('$PATH/datasets/infancy_brain_MRI')

if hasattr(model.hparams, "img_files"):
    img_files = model.hparams.img_files  
else:
    config = json.load(open(data_path / "dataset.json"))
    config["img_files"]
img_files = [f.replace(".npy", ".nii.gz") for f in img_files]

tfs_train, tfs_val = get_transforms(model.hparams)
tfs_used = tfs_val
tta_nr = 1

dataset_val = NiftiDataset(data_path, subjects_test, labels_test, wm_test, icv_test, ratio_test,
                            transform=tfs_used,
                            dim=model.hparams.dim,
                            nr_slices=model.hparams.nr_slices,
                            multi_orientation=model.hparams.multi_orientation,
                            crop_size=model.hparams.crop_size,
                            slice_subset=model.hparams.slice_subset,
                            img_files=img_files,
                            loss=model.hparams.loss,
                            nr_classes=model.hparams.nr_classes,
                            deterministic=True,
                            tiles=model.hparams.tiles,
                            file_loader=nifti_loader,
                            tta_nr=tta_nr,
                            tiles_subsample=model.hparams.tiles_subsample,
                            tiles_start=model.hparams.tiles_start,
                            zoom=model.hparams.zoom,
                            slice_orientation=model.hparams.slice_orientation)

# Per default DataLoader not doing any shuffling even with num_workers > 1
loader_val = DataLoader(dataset_val, batch_size=batch_size, num_workers=10, pin_memory=True)

data = next(iter(loader_val))
samples, y, a, b, c = data

encoder = model.backbone
layer_names = ['Layer1', 'Layer2', 'Layer3']
layers = [encoder.Conv_1, encoder.Conv_2, encoder.Conv_3]

i = 69
sample = samples[i]
wm = torch.tensor(a[i].reshape(-1, 1).float()).to(device)
icv = torch.tensor(c[i].reshape(-1, 1).float()).to(device)
info = torch.concat((wm, icv), dim=1)

slice_idx = 60
input_tensor = torch.tensor(sample).unsqueeze(0).to(device) 
input_for_viz = input_tensor.squeeze(0).permute(1, 2, 3, 0).cpu().numpy() 

img_slice1 = input_for_viz[:, :, slice_idx, 0:1]
img_slice = img_slice1
img_norm = (img_slice - img_slice.min()) / (img_slice.max() - img_slice.min() + 1e-6)

# figure setup
fig, axes = plt.subplots(1, len(layers) + 1, figsize=(3 * (len(layers) + 1), 4))

fig.text(0.02, 0.5, 'T2w', va='center', rotation='vertical', fontsize=25)

axes[0].imshow(np.rot90(img_norm), cmap='gray')
axes[0].axis("off")

for i, layer in enumerate(layers):
    cam = GradCAM(model=model, target_layers=[layer])
    grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0]  # [D, H, W]
    cam_slice = grayscale_cam[:, :, slice_idx]
    overlay = show_cam_on_image(img_norm, cam_slice, use_rgb=True, image_weight=0.2)

    axes[i + 1].imshow(np.rot90(overlay), cmap='jet')
    axes[i + 1].axis("off")

plt.tight_layout(rect=[0.04, 0, 1, 1])
plt.savefig(f"T2w.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close() 

In [ ]:
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


best_weights_path = '$PATH/RI_anatomical_prior/log/basic_0619/t1t2/version_3/checkpoints/epoch=021.ckpt'

def _add_missing_hparams(model):
    hparams_all = {v[0]: v[0] for k, v in args_short_to_long.items()}  # transform to dict param_long_name:default_value
    for param, default_value in hparams_all.items():
        if param not in model.hparams:
            if "lightning_quiet" not in os.environ:
                print(f"Missing hparam '{param}' in stored model. Adding with value {default_value}.")
            model.hparams[param] = default_value
    return model


batch_size = 70
device = torch.device('cuda:0')

models = []
model = LitClassifier.load_from_checkpoint(best_weights_path)
model = _add_missing_hparams(model)
model.eval().to(device)
models.append(model)
model = models[0] 

subjects_test = model.hparams.subjects_test
labels_test = model.hparams.labels_test
wm_test = model.hparams.wm_test
icv_test = model.hparams.icv_test
ratio_test = model.hparams.ratio_test

data_path = Path('$PATH/datasets/infancy_brain_MRI')

if hasattr(model.hparams, "img_files"):
    img_files = model.hparams.img_files  
else:
    config = json.load(open(data_path / "dataset.json"))
    config["img_files"]
img_files = [f.replace(".npy", ".nii.gz") for f in img_files]

tfs_train, tfs_val = get_transforms(model.hparams)
tfs_used = tfs_val
tta_nr = 1

dataset_val = NiftiDataset(data_path, subjects_test, labels_test, wm_test, icv_test, ratio_test,
                            transform=tfs_used,
                            dim=model.hparams.dim,
                            nr_slices=model.hparams.nr_slices,
                            multi_orientation=model.hparams.multi_orientation,
                            crop_size=model.hparams.crop_size,
                            slice_subset=model.hparams.slice_subset,
                            img_files=img_files,
                            loss=model.hparams.loss,
                            nr_classes=model.hparams.nr_classes,
                            deterministic=True,
                            tiles=model.hparams.tiles,
                            file_loader=nifti_loader,
                            tta_nr=tta_nr,
                            tiles_subsample=model.hparams.tiles_subsample,
                            tiles_start=model.hparams.tiles_start,
                            zoom=model.hparams.zoom,
                            slice_orientation=model.hparams.slice_orientation)

# Per default DataLoader not doing any shuffling even with num_workers > 1
loader_val = DataLoader(dataset_val, batch_size=batch_size, num_workers=10, pin_memory=True)

data = next(iter(loader_val))
samples, y, a, b, c = data

encoder = model.backbone
layer_names = ['Layer1', 'Layer2', 'Layer3']
layers = [encoder.Conv_1, encoder.Conv_2, encoder.Conv_3]

i = 69
sample = samples[i]
wm = torch.tensor(a[i].reshape(-1, 1).float()).to(device)
icv = torch.tensor(c[i].reshape(-1, 1).float()).to(device)
info = torch.concat((wm, icv), dim=1)

slice_idx = 60
input_tensor = torch.tensor(sample).unsqueeze(0).to(device) 
input_for_viz = input_tensor.squeeze(0).permute(1, 2, 3, 0).cpu().numpy() 

img_slice1 = input_for_viz[:, :, slice_idx, 0:1]
img_slice2 = input_for_viz[:, :, slice_idx, 1:2]
img_slice = img_slice1 + img_slice2
img_norm = (img_slice - img_slice.min()) / (img_slice.max() - img_slice.min() + 1e-6)

# figure setup
fig, axes = plt.subplots(1, len(layers) + 1, figsize=(3 * (len(layers) + 1), 4))

fig.text(0.02, 0.5, 'T1w+T2w', va='center', rotation='vertical', fontsize=25)

axes[0].imshow(np.rot90(img_norm), cmap='gray')
axes[0].axis("off")

for i, layer in enumerate(layers):
    cam = GradCAM(model=model, target_layers=[layer])
    grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0]  # [D, H, W]
    cam_slice = grayscale_cam[:, :, slice_idx]
    overlay = show_cam_on_image(img_norm, cam_slice, use_rgb=True, image_weight=0.05)

    axes[i + 1].imshow(np.rot90(overlay), cmap='jet')
    axes[i + 1].axis("off")

plt.tight_layout(rect=[0.04, 0, 1, 1])
plt.savefig(f"T1T2.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close() 

In [ ]:
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


best_weights_path = '$PATH/RI_anatomical_prior/log/basic_0619/ratio/version_3/checkpoints/epoch=016.ckpt'

def _add_missing_hparams(model):
    hparams_all = {v[0]: v[0] for k, v in args_short_to_long.items()}  # transform to dict param_long_name:default_value
    for param, default_value in hparams_all.items():
        if param not in model.hparams:
            if "lightning_quiet" not in os.environ:
                print(f"Missing hparam '{param}' in stored model. Adding with value {default_value}.")
            model.hparams[param] = default_value
    return model


batch_size = 70
device = torch.device('cuda:0')

models = []
model = LitClassifier.load_from_checkpoint(best_weights_path)
model = _add_missing_hparams(model)
model.eval().to(device)
models.append(model)
model = models[0] 

subjects_test = model.hparams.subjects_test
labels_test = model.hparams.labels_test
wm_test = model.hparams.wm_test
icv_test = model.hparams.icv_test
ratio_test = model.hparams.ratio_test

data_path = Path('$PATH/datasets/infancy_brain_MRI')

if hasattr(model.hparams, "img_files"):
    img_files = model.hparams.img_files  
else:
    config = json.load(open(data_path / "dataset.json"))
    config["img_files"]
img_files = [f.replace(".npy", ".nii.gz") for f in img_files]

tfs_train, tfs_val = get_transforms(model.hparams)
tfs_used = tfs_val
tta_nr = 1

dataset_val = NiftiDataset(data_path, subjects_test, labels_test, wm_test, icv_test, ratio_test,
                            transform=tfs_used,
                            dim=model.hparams.dim,
                            nr_slices=model.hparams.nr_slices,
                            multi_orientation=model.hparams.multi_orientation,
                            crop_size=model.hparams.crop_size,
                            slice_subset=model.hparams.slice_subset,
                            img_files=img_files,
                            loss=model.hparams.loss,
                            nr_classes=model.hparams.nr_classes,
                            deterministic=True,
                            tiles=model.hparams.tiles,
                            file_loader=nifti_loader,
                            tta_nr=tta_nr,
                            tiles_subsample=model.hparams.tiles_subsample,
                            tiles_start=model.hparams.tiles_start,
                            zoom=model.hparams.zoom,
                            slice_orientation=model.hparams.slice_orientation)

# Per default DataLoader not doing any shuffling even with num_workers > 1
loader_val = DataLoader(dataset_val, batch_size=batch_size, num_workers=10, pin_memory=True)

data = next(iter(loader_val))
samples, y, a, b, c = data

encoder = model.backbone
layer_names = ['Layer1', 'Layer2', 'Layer3']
layers = [encoder.Conv_1, encoder.Conv_2, encoder.Conv_3]

i = 69
sample = samples[i]
wm = torch.tensor(a[i].reshape(-1, 1).float()).to(device)
icv = torch.tensor(c[i].reshape(-1, 1).float()).to(device)
info = torch.concat((wm, icv), dim=1)

slice_idx = 60
input_tensor = torch.tensor(sample).unsqueeze(0).to(device) 
input_for_viz = input_tensor.squeeze(0).permute(1, 2, 3, 0).cpu().numpy() 

img_slice1 = input_for_viz[:, :, slice_idx, 0:1]
img_slice = img_slice1
img_norm = (img_slice - img_slice.min()) / (img_slice.max() - img_slice.min() + 1e-6)

# figure setup
fig, axes = plt.subplots(1, len(layers) + 1, figsize=(3 * (len(layers) + 1), 4))

fig.text(0.02, 0.5, 'RI', va='center', rotation='vertical', fontsize=25)

axes[0].imshow(np.rot90(img_norm), cmap='gray')
axes[0].axis("off")

for i, layer in enumerate(layers):
    cam = GradCAM(model=model, target_layers=[layer])
    grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0]  # [D, H, W]
    cam_slice = grayscale_cam[:, :, slice_idx]
    overlay = show_cam_on_image(img_norm, cam_slice, use_rgb=True, image_weight=0.05)

    axes[i + 1].imshow(np.rot90(overlay), cmap='jet')
    axes[i + 1].axis("off")

plt.tight_layout(rect=[0.04, 0, 1, 1])
plt.savefig(f"RI.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close() 

In [ ]:
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


best_weights_path = '$PATH/RI_anatomical_prior/log/basic_0619/t1t2ratio/version_5/checkpoints/epoch=011.ckpt'

def _add_missing_hparams(model):
    hparams_all = {v[0]: v[0] for k, v in args_short_to_long.items()}  # transform to dict param_long_name:default_value
    for param, default_value in hparams_all.items():
        if param not in model.hparams:
            if "lightning_quiet" not in os.environ:
                print(f"Missing hparam '{param}' in stored model. Adding with value {default_value}.")
            model.hparams[param] = default_value
    return model


batch_size = 70
device = torch.device('cuda:0')

models = []
model = LitClassifier.load_from_checkpoint(best_weights_path)
model = _add_missing_hparams(model)
model.eval().to(device)
models.append(model)
model = models[0] 

subjects_test = model.hparams.subjects_test
labels_test = model.hparams.labels_test
wm_test = model.hparams.wm_test
icv_test = model.hparams.icv_test
ratio_test = model.hparams.ratio_test

data_path = Path('$PATH/datasets/infancy_brain_MRI')

if hasattr(model.hparams, "img_files"):
    img_files = model.hparams.img_files  
else:
    config = json.load(open(data_path / "dataset.json"))
    config["img_files"]
img_files = [f.replace(".npy", ".nii.gz") for f in img_files]

tfs_train, tfs_val = get_transforms(model.hparams)
tfs_used = tfs_val
tta_nr = 1

dataset_val = NiftiDataset(data_path, subjects_test, labels_test, wm_test, icv_test, ratio_test,
                            transform=tfs_used,
                            dim=model.hparams.dim,
                            nr_slices=model.hparams.nr_slices,
                            multi_orientation=model.hparams.multi_orientation,
                            crop_size=model.hparams.crop_size,
                            slice_subset=model.hparams.slice_subset,
                            img_files=img_files,
                            loss=model.hparams.loss,
                            nr_classes=model.hparams.nr_classes,
                            deterministic=True,
                            tiles=model.hparams.tiles,
                            file_loader=nifti_loader,
                            tta_nr=tta_nr,
                            tiles_subsample=model.hparams.tiles_subsample,
                            tiles_start=model.hparams.tiles_start,
                            zoom=model.hparams.zoom,
                            slice_orientation=model.hparams.slice_orientation)

# Per default DataLoader not doing any shuffling even with num_workers > 1
loader_val = DataLoader(dataset_val, batch_size=batch_size, num_workers=10, pin_memory=True)

data = next(iter(loader_val))
samples, y, a, b, c = data

encoder = model.backbone
layer_names = ['Layer1', 'Layer2', 'Layer3']
layers = [encoder.Conv_1, encoder.Conv_2, encoder.Conv_3]

i = 69
sample = samples[i]
wm = torch.tensor(a[i].reshape(-1, 1).float()).to(device)
icv = torch.tensor(c[i].reshape(-1, 1).float()).to(device)
info = torch.concat((wm, icv), dim=1)

slice_idx = 60
input_tensor = torch.tensor(sample).unsqueeze(0).to(device) 
input_for_viz = input_tensor.squeeze(0).permute(1, 2, 3, 0).cpu().numpy() 

img_slice1 = input_for_viz[:, :, slice_idx, 0:1]
img_slice2 = input_for_viz[:, :, slice_idx, 1:2]
img_slice3 = input_for_viz[:, :, slice_idx, 2:3]
img_slice = img_slice1 + img_slice2 + 5 * img_slice3
img_norm = (img_slice - img_slice.min()) / (img_slice.max() - img_slice.min() + 1e-6)

# # figure setup




### RI specific

In [ ]:
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


best_weights_path = '$PATH/RI_anatomical_prior/log/basic_0619/ratio/version_3/checkpoints/epoch=016.ckpt'

def _add_missing_hparams(model):
    hparams_all = {v[0]: v[0] for k, v in args_short_to_long.items()}  # transform to dict param_long_name:default_value
    for param, default_value in hparams_all.items():
        if param not in model.hparams:
            if "lightning_quiet" not in os.environ:
                print(f"Missing hparam '{param}' in stored model. Adding with value {default_value}.")
            model.hparams[param] = default_value
    return model


batch_size = 129
device = torch.device('cuda:0')

models = []
model = LitClassifier.load_from_checkpoint(best_weights_path)
model = _add_missing_hparams(model)
model.eval().to(device)
models.append(model)
model = models[0] 

subjects_test = model.hparams.subjects_test
labels_test = model.hparams.labels_test
wm_test = model.hparams.wm_test
icv_test = model.hparams.icv_test
ratio_test = model.hparams.ratio_test

data_path = Path('$PATH/datasets/infancy_brain_MRI')

if hasattr(model.hparams, "img_files"):
    img_files = model.hparams.img_files  
else:
    config = json.load(open(data_path / "dataset.json"))
    config["img_files"]
img_files = [f.replace(".npy", ".nii.gz") for f in img_files]

tfs_train, tfs_val = get_transforms(model.hparams)
tfs_used = tfs_val
tta_nr = 1

dataset_val = NiftiDataset(data_path, subjects_test, labels_test, wm_test, icv_test, ratio_test,
                            transform=tfs_used,
                            dim=model.hparams.dim,
                            nr_slices=model.hparams.nr_slices,
                            multi_orientation=model.hparams.multi_orientation,
                            crop_size=model.hparams.crop_size,
                            slice_subset=model.hparams.slice_subset,
                            img_files=img_files,
                            loss=model.hparams.loss,
                            nr_classes=model.hparams.nr_classes,
                            deterministic=True,
                            tiles=model.hparams.tiles,
                            file_loader=nifti_loader,
                            tta_nr=tta_nr,
                            tiles_subsample=model.hparams.tiles_subsample,
                            tiles_start=model.hparams.tiles_start,
                            zoom=model.hparams.zoom,
                            slice_orientation=model.hparams.slice_orientation)

# Per default DataLoader not doing any shuffling even with num_workers > 1
loader_val = DataLoader(dataset_val, batch_size=batch_size, num_workers=10, pin_memory=True)

data = next(iter(loader_val))
samples, y, a, b, c = data




In [ ]:
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


best_weights_path = '$PATH/RI_anatomical_prior/log/basic_0619/ratio/version_3/checkpoints/epoch=016.ckpt'

def _add_missing_hparams(model):
    hparams_all = {v[0]: v[0] for k, v in args_short_to_long.items()}  # transform to dict param_long_name:default_value
    for param, default_value in hparams_all.items():
        if param not in model.hparams:
            if "lightning_quiet" not in os.environ:
                print(f"Missing hparam '{param}' in stored model. Adding with value {default_value}.")
            model.hparams[param] = default_value
    return model


batch_size = 129
device = torch.device('cuda:0')

models = []
model = LitClassifier.load_from_checkpoint(best_weights_path)
model = _add_missing_hparams(model)
model.eval().to(device)
models.append(model)
model = models[0] 

subjects_test = model.hparams.subjects_test
labels_test = model.hparams.labels_test
wm_test = model.hparams.wm_test
icv_test = model.hparams.icv_test
ratio_test = model.hparams.ratio_test

data_path = Path('$PATH/datasets/infancy_brain_MRI')

if hasattr(model.hparams, "img_files"):
    img_files = model.hparams.img_files  
else:
    config = json.load(open(data_path / "dataset.json"))
    config["img_files"]
img_files = [f.replace(".npy", ".nii.gz") for f in img_files]

tfs_train, tfs_val = get_transforms(model.hparams)
tfs_used = tfs_val
tta_nr = 1

dataset_val = NiftiDataset(data_path, subjects_test, labels_test, wm_test, icv_test, ratio_test,
                            transform=tfs_used,
                            dim=model.hparams.dim,
                            nr_slices=model.hparams.nr_slices,
                            multi_orientation=model.hparams.multi_orientation,
                            crop_size=model.hparams.crop_size,
                            slice_subset=model.hparams.slice_subset,
                            img_files=img_files,
                            loss=model.hparams.loss,
                            nr_classes=model.hparams.nr_classes,
                            deterministic=True,
                            tiles=model.hparams.tiles,
                            file_loader=nifti_loader,
                            tta_nr=tta_nr,
                            tiles_subsample=model.hparams.tiles_subsample,
                            tiles_start=model.hparams.tiles_start,
                            zoom=model.hparams.zoom,
                            slice_orientation=model.hparams.slice_orientation)

# Per default DataLoader not doing any shuffling even with num_workers > 1
loader_val = DataLoader(dataset_val, batch_size=batch_size, num_workers=10, pin_memory=True)

data = next(iter(loader_val))
samples, y, a, b, c = data




In [ ]:
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


best_weights_path = '$PATH/RI_anatomical_prior/log/basic_0619/ratio/version_3/checkpoints/epoch=016.ckpt'

def _add_missing_hparams(model):
    hparams_all = {v[0]: v[0] for k, v in args_short_to_long.items()}  # transform to dict param_long_name:default_value
    for param, default_value in hparams_all.items():
        if param not in model.hparams:
            if "lightning_quiet" not in os.environ:
                print(f"Missing hparam '{param}' in stored model. Adding with value {default_value}.")
            model.hparams[param] = default_value
    return model


batch_size = 70
device = torch.device('cuda:0')

models = []
model = LitClassifier.load_from_checkpoint(best_weights_path)
model = _add_missing_hparams(model)
model.eval().to(device)
models.append(model)
model = models[0] 

subjects_test = model.hparams.subjects_test
labels_test = model.hparams.labels_test
wm_test = model.hparams.wm_test
icv_test = model.hparams.icv_test
ratio_test = model.hparams.ratio_test

data_path = Path('$PATH/datasets/infancy_brain_MRI')

if hasattr(model.hparams, "img_files"):
    img_files = model.hparams.img_files  
else:
    config = json.load(open(data_path / "dataset.json"))
    config["img_files"]
img_files = [f.replace(".npy", ".nii.gz") for f in img_files]

tfs_train, tfs_val = get_transforms(model.hparams)
tfs_used = tfs_val
tta_nr = 1

dataset_val = NiftiDataset(data_path, subjects_test, labels_test, wm_test, icv_test, ratio_test,
                            transform=tfs_used,
                            dim=model.hparams.dim,
                            nr_slices=model.hparams.nr_slices,
                            multi_orientation=model.hparams.multi_orientation,
                            crop_size=model.hparams.crop_size,
                            slice_subset=model.hparams.slice_subset,
                            img_files=img_files,
                            loss=model.hparams.loss,
                            nr_classes=model.hparams.nr_classes,
                            deterministic=True,
                            tiles=model.hparams.tiles,
                            file_loader=nifti_loader,
                            tta_nr=tta_nr,
                            tiles_subsample=model.hparams.tiles_subsample,
                            tiles_start=model.hparams.tiles_start,
                            zoom=model.hparams.zoom,
                            slice_orientation=model.hparams.slice_orientation)

# Per default DataLoader not doing any shuffling even with num_workers > 1
loader_val = DataLoader(dataset_val, batch_size=batch_size, num_workers=10, pin_memory=True)

data = next(iter(loader_val))
samples, y, a, b, c = data

encoder = model.backbone
layer_names = ['Layer1', 'Layer2', 'Layer3']
layers = [encoder.Conv_1, encoder.Conv_2, encoder.Conv_3]

i = 69
sample = samples[i]
wm = torch.tensor(a[i].reshape(-1, 1).float()).to(device)
icv = torch.tensor(c[i].reshape(-1, 1).float()).to(device)
info = torch.concat((wm, icv), dim=1)

slice_idx = 60
input_tensor = torch.tensor(sample).unsqueeze(0).to(device) 
input_for_viz = input_tensor.squeeze(0).permute(1, 2, 3, 0).cpu().numpy() 

img_slice1 = input_for_viz[:, :, slice_idx, 0:1]
img_slice = img_slice1
img_norm = (img_slice - img_slice.min()) / (img_slice.max() - img_slice.min() + 1e-6)

num_slices = 30
num_cols = 5
mid_slice = img_norm.shape[2] // 2
step = max(1, img_norm.shape[2] // num_slices)
slice_indices = np.linspace(mid_slice - (num_slices // 2) * step, 
                            mid_slice + (num_slices // 2) * step, 
                            num_slices, dtype=int)

num_rows = int(np.ceil(num_slices / num_cols))

fig, axes = plt.subplots(num_rows, num_cols, figsize=(3*num_cols, 3*num_rows))
axes = axes.flatten()

for i, idx in enumerate(slice_indices):
    axes[i].imshow(data[:, :, idx].T, cmap="gray", origin="lower")
    axes[i].set_title(f"Slice {idx}", fontsize=10)
    axes[i].axis("off")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle('Original axial', fontsize=14)
plt.tight_layout()
plt.show()


for i, layer in enumerate(layers):
    cam = GradCAM(model=model, target_layers=[layer])
    grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0]  # [D, H, W]
    cam_slice = grayscale_cam[:, :, slice_idx]
    overlay = show_cam_on_image(img_norm, cam_slice, use_rgb=True, image_weight=0.05)

    axes[i + 1].imshow(np.rot90(overlay), cmap='jet')
    axes[i + 1].axis("off")

plt.tight_layout(rect=[0.04, 0, 1, 1])
plt.savefig(f"RI.png", dpi=300, bbox_inches='tight')
plt.show()
plt.close() 

num_slices = 30
num_cols = 5
mid_slice = img_norm.shape[2] // 2
step = max(1, img_norm.shape[2] // num_slices)
slice_indices = np.linspace(mid_slice - (num_slices // 2) * step, 
                            mid_slice + (num_slices // 2) * step, 
                            num_slices, dtype=int)

num_rows = int(np.ceil(num_slices / num_cols))

fig, axes = plt.subplots(num_rows, num_cols, figsize=(3*num_cols, 3*num_rows))
axes = axes.flatten()

for i, idx in enumerate(slice_indices):
    axes[i].imshow(data[:, :, idx].T, cmap="gray", origin="lower")
    axes[i].set_title(f"Slice {idx}", fontsize=10)
    axes[i].axis("off")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle('Original axial', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

import torch
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from monai.transforms import NormalizeIntensity

from lit_classifier import LitClassifier
import warnings
import pytorch_lightning as pl
import random
warnings.filterwarnings("ignore")

seed = 1234
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True  # CuDNN deterministic
torch.backends.cudnn.benchmark = False  # CuDNN benchmark off
torch.random.manual_seed(seed)
pl.seed_everything(seed)


def normalize_image(data):
    min_val = np.min(data)
    max_val = np.max(data)
    return (data - min_val) / (max_val - min_val + 1e-6)

def plot_multi_view(title, raw, cam, view='axial', num_slices=10, num_cols=5):
    assert view in ['axial', 'sagittal', 'coronal']
    raw = normalize_image(raw)
    
    if view == 'axial':
        axis = 2
        name = '(a)'
    elif view == 'sagittal':
        axis = 0
        name = '(c)'
    elif view == 'coronal':
        axis = 1
        name = '(b)'

    raw = np.moveaxis(raw, axis, 2)
    cam = np.moveaxis(cam, axis, 2)

    total_slices = raw.shape[2]
    start = int(total_slices * 0.15)
    end = int(total_slices * 0.9)
    valid_range = raw[:, :, start:end]

    slice_indices = np.linspace(start, end - 1, num_slices, dtype=int)
    num_rows = int(np.ceil(num_slices / num_cols))

    fig, axes = plt.subplots(num_rows, num_cols * 2, figsize=(3 * num_cols * 2, 3.7 * num_rows))

    if axes.ndim == 1:
        axes = axes[np.newaxis, :]

    axes = axes.reshape(num_rows, num_cols * 2)

    for i, idx in enumerate(slice_indices):
        row = i // num_cols
        col = (i % num_cols) * 2

        raw_slice = raw[:, :, idx]
        cam_slice = cam[:, :, idx]


        rgb_raw = np.expand_dims(raw_slice, axis=2).repeat(3, axis=2)
        overlay = show_cam_on_image(rgb_raw, cam_slice, use_rgb=True, image_weight=0.05)

        axes[row, col].imshow(np.rot90(raw_slice), cmap="gray")
        if view != 'sagittal':
            axes[row, col].imshow(np.rot90(raw_slice), cmap="gray")
        if view == 'sagittal':
            axes[row, col].imshow(np.fliplr(np.rot90(raw_slice)), cmap='gray')
        axes[row, col].set_title(f"Original {idx}", fontsize=15)
        axes[row, col].axis("off")

        axes[row, col + 1].imshow(np.rot90(overlay))
        if view != 'sagittal':
            axes[row, col + 1].imshow(np.rot90(overlay))
        if view == 'sagittal':
            axes[row, col + 1].imshow(np.fliplr(np.rot90(overlay)))
        axes[row, col + 1].set_title(f"Grad-CAM {idx}", fontsize=15)
        axes[row, col + 1].axis("off")

    for j in range(i + 1, num_rows * num_cols):
        axes[j // (num_cols * 2), j % (num_cols * 2)].axis("off")

    plt.suptitle(f"{name} {view.title()} View", fontsize=25)
    plt.tight_layout()
    plt.show()


image_path = "$PATH/datasets/infancy_brain_MRI/sample_0001/ratio_regist_15_modified.nii.gz" ## 669 (16)
img = nib.load(image_path)
data = img.get_fdata()

normalize_all = NormalizeIntensity(nonzero=False)
data_norm = normalize_all(np.expand_dims(data, 0)).squeeze().numpy()

device = torch.device('cuda:0')
best_weights_path = '$PATH/RI_anatomical_prior/log/basic_0619/ratio/version_3/checkpoints/epoch=016.ckpt'
model = LitClassifier.load_from_checkpoint(best_weights_path).eval().to(device)
cam_layer = model.backbone.Conv_2
cam = GradCAM(model=model, target_layers=[cam_layer])

input_tensor = torch.tensor(data_norm).unsqueeze(0).unsqueeze(0).to(device)  # shape: [1, 1, H, W, D]
grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0]  # shape: [H, W, D]


for view in ['axial', 'coronal', 'sagittal']:
    plot_multi_view("Brain MRI", data_norm, grayscale_cam, view=view)
